# Call Red Online (HSE) APIs

Simple notebook — no project package imports. Uses standard library + optional `requests`.

On corporate networks, TLS verify is turned **off** (`verify=False`) to avoid `CERTIFICATE_VERIFY_FAILED`.

In [1]:
import json
import ssl
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

# --- fill / confirm these ---
BASE_URL = "https://apigw.ct-test.hse-compliance.net"
API_KEY = "9f5ef52b-b98d-4cdb-9672-e423cc9051d5"
API_KEY_HEADER = "X-ROL-API-KEY"
SITE_ID = "112087"
USER_ID = "584646"
VERIFY_SSL = False  # False on corp laptop if SSL interception breaks certs

OUT_DIR = Path("../samples/rol")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ENDPOINTS = {
    "list_sites": {
        "path": "/v2/secure-clients/legapi-general/sites",
        "params": {},
    },
    "list_users_by_site": {
        "path": f"/v2/secure-clients/legapi-general/users/sites/{SITE_ID}",
        "params": {},
    },
    "list_mapping_references": {
        "path": "/v2/secure-clients/tasks/configs/references",
        "params": {"user_id": USER_ID},
    },
    "list_tasks_by_user": {
        "path": "/v2/secure-clients/tasks",
        "params": {"user_id": USER_ID},
    },
}

In [2]:
def rol_get(path: str, params: dict | None = None):
    url = BASE_URL.rstrip("/") + path
    if params:
        url = f"{url}?{urlencode(params)}"
    headers = {
        "Accept": "application/json",
        API_KEY_HEADER: API_KEY,
    }
    req = Request(url, headers=headers, method="GET")
    context = None if VERIFY_SSL else ssl._create_unverified_context()
    print("GET", url)
    with urlopen(req, timeout=60, context=context) as resp:
        body = resp.read().decode("utf-8")
        return json.loads(body) if body else None


def save_json(name: str, payload):
    path = OUT_DIR / f"{name}_raw.json"
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    print("saved", path.resolve())
    return path

## Call one endpoint (sites)

In [3]:
sites = rol_get(ENDPOINTS["list_sites"]["path"])
save_json("list_sites", sites)
sites if not isinstance(sites, list) else sites[:2]

GET https://apigw.ct-test.hse-compliance.net/v2/secure-clients/legapi-general/sites
saved C:\Users\CRANB1\Cursor\Cursor\redonline-cdf-ingest\samples\rol\list_sites_raw.json


{'sites': [{'id_site': 112084,
   'id_parent': None,
   'name': 'Celanese',
   'id_site_type': 1,
   'id_country': None,
   'id_ref_status': 1,
   'code_iso': '',
   'country_name': ''},
  {'id_site': 112087,
   'id_parent': 1579560,
   'name': 'GEL - Geleen, Netherlands',
   'id_site_type': 6,
   'id_country': 47,
   'id_ref_status': 1,
   'code_iso': 'NL',
   'country_name': 'Netherlands'},
  {'id_site': 1579306,
   'id_parent': 1579545,
   'name': 'CLK - Clear Lake, Texas',
   'id_site_type': 6,
   'id_country': 29,
   'id_ref_status': 1,
   'code_iso': 'US',
   'country_name': 'United States of America'},
  {'id_site': 1579310,
   'id_parent': 1579545,
   'name': 'BCH - Boucherville, Quebec',
   'id_site_type': 6,
   'id_country': 30,
   'id_ref_status': 1,
   'code_iso': 'CA',
   'country_name': 'Canada'},
  {'id_site': 1579313,
   'id_parent': 1579413,
   'name': 'NAN - Nanjing Mfg',
   'id_site_type': 6,
   'id_country': 34,
   'id_ref_status': 1,
   'code_iso': 'CN',
   'countr

## Call all endpoints and save responses

In [4]:
results = {}
for name, cfg in ENDPOINTS.items():
    try:
        payload = rol_get(cfg["path"], cfg.get("params") or {})
        save_json(name, payload)
        results[name] = {
            "ok": True,
            "type": type(payload).__name__,
            "count": len(payload) if isinstance(payload, list) else None,
            "top_keys": list(payload.keys())[:20] if isinstance(payload, dict) else None,
        }
    except Exception as exc:
        results[name] = {"ok": False, "error": f"{type(exc).__name__}: {exc}"}
        print("FAILED", name, results[name]["error"])

results

GET https://apigw.ct-test.hse-compliance.net/v2/secure-clients/legapi-general/sites
saved C:\Users\CRANB1\Cursor\Cursor\redonline-cdf-ingest\samples\rol\list_sites_raw.json
GET https://apigw.ct-test.hse-compliance.net/v2/secure-clients/legapi-general/users/sites/112087
saved C:\Users\CRANB1\Cursor\Cursor\redonline-cdf-ingest\samples\rol\list_users_by_site_raw.json
GET https://apigw.ct-test.hse-compliance.net/v2/secure-clients/tasks/configs/references?user_id=584646
saved C:\Users\CRANB1\Cursor\Cursor\redonline-cdf-ingest\samples\rol\list_mapping_references_raw.json
GET https://apigw.ct-test.hse-compliance.net/v2/secure-clients/tasks?user_id=584646
saved C:\Users\CRANB1\Cursor\Cursor\redonline-cdf-ingest\samples\rol\list_tasks_by_user_raw.json


{'list_sites': {'ok': True,
  'type': 'dict',
  'count': None,
  'top_keys': ['sites', 'sync_date']},
 'list_users_by_site': {'ok': True,
  'type': 'dict',
  'count': None,
  'top_keys': ['users']},
 'list_mapping_references': {'ok': True,
  'type': 'dict',
  'count': None,
  'top_keys': ['success', 'message', 'data', 'debug']},
 'list_tasks_by_user': {'ok': True,
  'type': 'dict',
  'count': None,
  'top_keys': ['success', 'message', 'data', 'meta', 'debug']}}

## Inspect keys for RAW table design

If the payload is a list of objects, these keys become candidate RAW columns.

In [5]:
def first_object(payload):
    if isinstance(payload, list) and payload and isinstance(payload[0], dict):
        return payload[0]
    if isinstance(payload, dict):
        for key in ("data", "items", "results", "value"):
            val = payload.get(key)
            if isinstance(val, list) and val and isinstance(val[0], dict):
                return val[0]
            if isinstance(val, dict):
                return val
        return payload
    return None


for name in ENDPOINTS:
    path = OUT_DIR / f"{name}_raw.json"
    if not path.exists():
        print(name, "-> no saved file")
        continue
    payload = json.loads(path.read_text(encoding="utf-8"))
    sample = first_object(payload)
    print("\n===", name, "===")
    if sample is None:
        print("could not find object sample; payload type:", type(payload).__name__)
    else:
        for key, value in sample.items():
            print(f"  {key}: {type(value).__name__} = {repr(value)[:80]}")


=== list_sites ===
  sites: list = [{'id_site': 112084, 'id_parent': None, 'name': 'Celanese', 'id_site_type': 1, '
  sync_date: str = '2026-10-04T16:41:27+02:00'

=== list_users_by_site ===
  users: list = [{'id_user': 156028, 'firstname': 'Monika', 'lastname': 'Hess', 'user_name': 'Mo

=== list_mapping_references ===
  types: list = [{'label': 'types.free', 'reference_type_id': 1, 'value': 1}, {'label': 'types.a
  categories: list = [{'label': 'categories.non_applicable', 'value': 1}, {'label': 'categories.corre
  priorities: list = [{'label': 'priorities.none', 'value': 0}, {'label': 'priorities.low', 'value': 
  scopes: list = [{'label': 'scopes.mono_site', 'value': 1}, {'label': 'scopes.multi_sites', 'val
  states: list = [{'label': 'states.opened', 'value': 2}, {'label': 'states.done', 'value': 6}, {
  occurrence_statuses: list = [{'label': 'occurrence_statuses.new', 'next_status_ids': [2, 7, 8], 'value': 1},
  reference_types: list = [{'label': 'reference_types.free', 'module_i